In [1]:
!pip install requests python-Levenshtein unidecode

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 153.3/153.3 kB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.8/235.8 kB 29.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 135.1 MB/s eta 0:00:00


In [2]:
import re
import json
import unicodedata
import Levenshtein
from unidecode import unidecode
from itertools import combinations
import requests

In [4]:
# Paso 1: Crear dataset de variantes de nombres
author_variants = [
    # Gabriel García Márquez
    {"id": "A001", "name": "García Márquez, Gabriel"},
    {"id": "A001", "name": "Gabriel García Márquez"},
    {"id": "A001", "name": "G. García Márquez"},
    {"id": "A001", "name": "Gabriel Garcia Marquez"},

    # Isabel Allende
    {"id": "A002", "name": "Allende, Isabel"},
    {"id": "A002", "name": "Isabel Allende"},
    {"id": "A002", "name": "I. Allende"},
    {"id": "A002", "name": "Isabel Allende Llona"},

    # Jorge Luis Borges
    {"id": "A003", "name": "Borges, Jorge Luis"},
    {"id": "A003", "name": "Jorge Luis Borges"},
    {"id": "A003", "name": "J. L. Borges"},
    {"id": "A003", "name": "Jorge L. Borges"},

    # Mario Vargas Llosa
    {"id": "A004", "name": "Vargas Llosa, Mario"},
    {"id": "A004", "name": "Mario Vargas Llosa"},
    {"id": "A004", "name": "M. Vargas Llosa"},
    {"id": "A004", "name": "Mario V. Llosa"},

    # Julio Cortázar
    {"id": "A005", "name": "Cortázar, Julio"},
    {"id": "A005", "name": "Julio Cortázar"},
    {"id": "A005", "name": "J. Cortázar"},
    {"id": "A005", "name": "Julio Cortazar"},

    # Stephen King
    {"id": "A006", "name": "King, Stephen"},
    {"id": "A006", "name": "Stephen King"},
    {"id": "A006", "name": "S. King"},
    {"id": "A006", "name": "Stephen Edwin King"},

    # J.K. Rowling
    {"id": "A007", "name": "Rowling, J. K."},
    {"id": "A007", "name": "J.K. Rowling"},
    {"id": "A007", "name": "Joanne Rowling"},
    {"id": "A007", "name": "Joanne K. Rowling"},

    # George R.R. Martin
    {"id": "A008", "name": "Martin, George R. R."},
    {"id": "A008", "name": "George R. R. Martin"},
    {"id": "A008", "name": "G.R.R. Martin"},
    {"id": "A008", "name": "George Raymond Richard Martin"},

    # Haruki Murakami
    {"id": "A009", "name": "Murakami, Haruki"},
    {"id": "A009", "name": "Haruki Murakami"},
    {"id": "A009", "name": "H. Murakami"},
    {"id": "A009", "name": "村上 春樹"},

    # Umberto Eco
    {"id": "A010", "name": "Eco, Umberto"},
    {"id": "A010", "name": "Umberto Eco"},
    {"id": "A010", "name": "U. Eco"},
    {"id": "A010", "name": "Umberto Nicola Eco"},
]

print(f"Dataset cargado: {len(author_variants)} variantes de {len(set(v['id'] for v in author_variants))} autores")

Dataset cargado: 40 variantes de 10 autores


In [5]:
# Paso 2: Normalizar nombres
def normalize_name(name: str) -> str:
    name = unidecode(name)                 # eliminar acentos
    name = name.lower()                   # minusculas
    name = re.sub(r'[^\w\s]', ' ', name)  # eliminar puntuacion
    name = re.sub(r'\s+', ' ', name).strip()  # normalizar espacios
    return name

# Prueba
for v in author_variants[:4]:
    print(f"Original: {v['name']:35} → Normalizado: {normalize_name(v['name'])}")

Original: García Márquez, Gabriel             → Normalizado: garcia marquez gabriel
Original: Gabriel García Márquez              → Normalizado: gabriel garcia marquez
Original: G. García Márquez                   → Normalizado: g garcia marquez
Original: Gabriel Garcia Marquez              → Normalizado: gabriel garcia marquez


In [6]:
# Paso 3: Tokenizar nombres
# Dividir en tokens (nombre, apellido, iniciales)
def tokenize_name(normalized: str) -> dict:
    tokens = normalized.split()
    words = [t for t in tokens if len(t) > 2]
    initials = [t for t in tokens if len(t) <= 2]
    return {
        "tokens": tokens,
        "words": words,
        "initials": initials
    }

test = normalize_name("J.C. García López")
print(f"Entrada: '{test}'")
print(f"Tokens: {tokenize_name(test)}")

Entrada: 'j c garcia lopez'
Tokens: {'tokens': ['j', 'c', 'garcia', 'lopez'], 'words': ['garcia', 'lopez'], 'initials': ['j', 'c']}


In [7]:
# Paso 4: Generar representación canónica
def canonical_form(name: str) -> str:
    norm = normalize_name(name)
    tokens = tokenize_name(norm)

    if not tokens["tokens"]:
        return ""
    sorted_tokens = sorted(tokens["tokens"], key=len, reverse=True)
    last_name = sorted_tokens[0]
    initials = [t[0] for t in sorted_tokens[1:] if t]

    canonical = last_name
    if initials:
        canonical += "_" + "".join(initials)

    return canonical

# prueba con A001
print("formas canonicas A001:")
for v in author_variants[:4]:
    print(f"  {v['name']:35} → {canonical_form(v['name'])}")

formas canonicas A001:
  García Márquez, Gabriel             → marquez_gg
  Gabriel García Márquez              → gabriel_mg
  G. García Márquez                   → marquez_gg
  Gabriel Garcia Marquez              → gabriel_mg


In [11]:
# Paso 5: Calcular similitud entre nombres
def name_similarity(name1: str, name2: str) -> float:
    c1 = canonical_form(name1)
    c2 = canonical_form(name2)
    sim_canonical = 0.0
    if c1 and c2:
        d = Levenshtein.distance(c1, c2)
        sim_canonical = 1 - (d / max(len(c1), len(c2)))

    # distancia levenshtein
    n1 = normalize_name(name1)
    n2 = normalize_name(name2)
    tokens1 = " ".join(sorted(n1.split()))
    tokens2 = " ".join(sorted(n2.split()))
    d2 = Levenshtein.distance(tokens1, tokens2)
    sim_tokens = 1 - (d2 / max(len(tokens1), len(tokens2)))

    return round(max(sim_canonical, sim_tokens), 3)

# umbral
SIMILARITY_THRESHOLD = 0.70

# prueba
pairs = list(combinations(author_variants[:4], 2))
for a, b in pairs:
    sim = name_similarity(a["name"], b["name"])
    match = "" if sim >= SIMILARITY_THRESHOLD else ""
    print(f"  {match} {sim:.2f} | '{a['name']}' vs '{b['name']}'")

   1.00 | 'García Márquez, Gabriel' vs 'Gabriel García Márquez'
   1.00 | 'García Márquez, Gabriel' vs 'G. García Márquez'
   1.00 | 'García Márquez, Gabriel' vs 'Gabriel Garcia Marquez'
   0.73 | 'Gabriel García Márquez' vs 'G. García Márquez'
   1.00 | 'Gabriel García Márquez' vs 'Gabriel Garcia Marquez'
   0.73 | 'G. García Márquez' vs 'Gabriel Garcia Marquez'


In [12]:
# Paso 6: Agrupar variantes (clustering de autores)
def cluster_authors(variants: list, threshold: float = SIMILARITY_THRESHOLD) -> dict:
    n = len(variants)
    clusters = {}
    assigned = {}

    cluster_id = 0
    for i in range(n):
        if i in assigned:
            continue
        clusters[cluster_id] = [variants[i]]
        assigned[i] = cluster_id
        for j in range(i + 1, n):
            if j in assigned:
                continue
            sim = name_similarity(variants[i]["name"], variants[j]["name"])
            if sim >= threshold:
                clusters[cluster_id].append(variants[j])
                assigned[j] = cluster_id
        cluster_id += 1

    return clusters

clusters = cluster_authors(author_variants)
print(f"Clusters encontrados: {len(clusters)}\n")
for cid, members in clusters.items():
    print(f"Cluster {cid}:")
    for m in members:
        print(f"  - [{m['id']}] {m['name']}")
    print()

Clusters encontrados: 14

Cluster 0:
  - [A001] García Márquez, Gabriel
  - [A001] Gabriel García Márquez
  - [A001] G. García Márquez
  - [A001] Gabriel Garcia Marquez

Cluster 1:
  - [A002] Allende, Isabel
  - [A002] Isabel Allende
  - [A002] I. Allende
  - [A002] Isabel Allende Llona

Cluster 2:
  - [A003] Borges, Jorge Luis
  - [A003] Jorge Luis Borges
  - [A003] J. L. Borges
  - [A003] Jorge L. Borges

Cluster 3:
  - [A004] Vargas Llosa, Mario
  - [A004] Mario Vargas Llosa
  - [A004] M. Vargas Llosa
  - [A004] Mario V. Llosa

Cluster 4:
  - [A005] Cortázar, Julio
  - [A005] Julio Cortázar
  - [A005] J. Cortázar
  - [A005] Julio Cortazar

Cluster 5:
  - [A006] King, Stephen
  - [A006] Stephen King
  - [A006] Stephen Edwin King

Cluster 6:
  - [A006] S. King

Cluster 7:
  - [A007] Rowling, J. K.
  - [A007] J.K. Rowling
  - [A007] Joanne Rowling
  - [A007] Joanne K. Rowling

Cluster 8:
  - [A008] Martin, George R. R.
  - [A008] George R. R. Martin
  - [A008] G.R.R. Martin

Cluster 9:

In [13]:
# Paso 7: Resolver autor de entrada

# detectar si la entrada tiene formato ORCID: XXXX-XXXX-XXXX-XXXX
def is_orcid(text: str) -> bool:
    return bool(re.match(r'^\d{4}-\d{4}-\d{4}-\d{3}[\dX]$', text.strip()))

# nombre
def resolve_author(input_text: str, known_variants: list) -> dict:
    input_text = input_text.strip()

    if is_orcid(input_text):
        return {"type": "orcid", "identifier": input_text, "resolved_name": None}
    best_match = None
    best_sim = 0.0
    for v in known_variants:
        sim = name_similarity(input_text, v["name"])
        if sim > best_sim:
            best_sim = sim
            best_match = v

    return {
        "type": "name",
        "input": input_text,
        "canonical": canonical_form(input_text),
        "best_match": best_match,
        "similarity": round(best_sim, 3),
        "resolved_id": best_match["id"] if best_sim >= SIMILARITY_THRESHOLD else "UNKNOWN"
    }

# test
tests = ["Juan Garcia", "0000-0002-1825-0097", "Ana Martinez", "xyz abc"]
for t in tests:
    r = resolve_author(t, author_variants)
    print(json.dumps(r, ensure_ascii=False, indent=2))
    print("---")

{
  "type": "name",
  "input": "Juan Garcia",
  "canonical": "garcia_j",
  "best_match": {
    "id": "A001",
    "name": "G. García Márquez"
  },
  "similarity": 0.5,
  "resolved_id": "UNKNOWN"
}
---
{
  "type": "orcid",
  "identifier": "0000-0002-1825-0097",
  "resolved_name": null
}
---
{
  "type": "name",
  "input": "Ana Martinez",
  "canonical": "martinez_a",
  "best_match": {
    "id": "A008",
    "name": "Martin, George R. R."
  },
  "similarity": 0.6,
  "resolved_id": "UNKNOWN"
}
---
{
  "type": "name",
  "input": "xyz abc",
  "canonical": "xyz_a",
  "best_match": {
    "id": "A001",
    "name": "G. García Márquez"
  },
  "similarity": 0.25,
  "resolved_id": "UNKNOWN"
}
---


In [14]:
# Paso 8: Consultar APIs de publicaciones con OpenAlex
def fetch_publications_openalex(name: str = None, orcid: str = None, max_results: int = 25) -> list:
    headers = {"User-Agent": "pipeline-t3/1.0 (tarea academica)"}

    # encontrar el autor en OpenAlex
    if orcid:
        # buscar por ORCID en el endpoint de autores
        author_url = f"https://api.openalex.org/authors?filter=orcid:{orcid}"
    else:
        author_url = f"https://api.openalex.org/authors?search={requests.utils.quote(name)}&per-page=1"

    r = requests.get(author_url, headers=headers, timeout=15)
    if r.status_code != 200:
        print(f"error al buscar autor: {r.status_code}")
        return []

    results = r.json().get("results", [])
    if not results:
        print(f"autor no encontrado en OpenAlex")
        return []

    author = results[0]
    openalex_id = author["id"].split("/")[-1]  # ej: "A5086198262"
    print(f"   OpenAlex ID: {openalex_id} | Nombre: {author['display_name']}")

    # obtener sus publicaciones
    works_url = f"https://api.openalex.org/works?filter=author.id:{openalex_id}&per-page={max_results}&sort=publication_year:desc"
    r2 = requests.get(works_url, headers=headers, timeout=15)
    if r2.status_code != 200:
        print(f"error al obtener works: {r2.status_code}")
        return []

    works = r2.json().get("results", [])
    print(f"   Works encontrados: {len(works)}")
    return works

# prueba
print("=== Test con ORCID ===")
raw_pubs = fetch_publications_openalex(orcid="0000-0002-9322-3515")  # Bengio real

print("\n=== Test con nombre ===")
raw_pubs2 = fetch_publications_openalex(name="Haruki Murakami")

=== Test con ORCID ===
   OpenAlex ID: A5086198262 | Nombre: Yoshua Bengio
   Works encontrados: 25

=== Test con nombre ===
   OpenAlex ID: A5009185410 | Nombre: Haruki Murakami
   Works encontrados: 25


In [16]:
# Paso 10: Normalizar autores de publicaciones
def normalize_publication_authors(publications: list) -> list:
    for pub in publications:
        pub["authors_canonical"] = [canonical_form(a) for a in pub["authors"]]
    return publications

publications = normalize_publication_authors(publications)
print("autores originales vs canónicos (primera pub):")
if publications:
    for orig, canon in zip(publications[0]["authors"][:3], publications[0]["authors_canonical"][:3]):
        print(f"  {orig:30} → {canon}")
else:
    print("no hay publicaciones.")

Autores originales vs canónicos (primera pub):
  Lior Horesh                    → horesh_l
  Ramón Nartallo-Kaluarachchi    → kaluarachchi_nr
  Shashanka Ubaru                → shashanka_u


In [18]:
# Paso 9: Parsear y estructurar publicaciones
def parse_publication(raw: dict) -> dict:
    authors = []
    for auth in raw.get("authorships", []):
        display = auth.get("author", {}).get("display_name", "")
        if display:
            authors.append(display)
    source_name = None
    primary_location = raw.get("primary_location")
    if primary_location and isinstance(primary_location, dict):
        source_obj = primary_location.get("source")
        if source_obj and isinstance(source_obj, dict):
            source_name = source_obj.get("display_name")

    return {
        "title": raw.get("title", ""),
        "authors": authors,
        "year": raw.get("publication_year"),
        "doi": raw.get("doi", "").replace("https://doi.org/", "") if raw.get("doi") else None,
        "abstract": raw.get("abstract_inverted_index"),  # normalizar abajo
        "source": source_name,
    }

def rebuild_abstract(inverted_index: dict) -> str:
    if not inverted_index:
        return ""
    word_positions = {}
    for word, positions in inverted_index.items():
        for pos in positions:
            word_positions[pos] = word
    return " ".join(word_positions[i] for i in sorted(word_positions))

# parsear
if not raw_pubs:
    print("no hay publicaciones para parsear, checar el autor o ORCID.")
else:
    publications = []
    for raw in raw_pubs:
        pub = parse_publication(raw)
        pub["abstract"] = rebuild_abstract(pub["abstract"])
        publications.append(pub)

    publications = normalize_publication_authors(publications)

    print(f"publicaciones parseadas: {len(publications)}\n")
    print("ejemplo:")
    if publications:
        print(json.dumps(publications[0], indent=2, ensure_ascii=False))
    else:
        print("no hay publicaciones.")

publicaciones parseadas: 25

ejemplo:
{
  "title": "Interpretable epistemic uncertainty decomposition in sequential generative models via polynomial chaos surrogates",
  "authors": [
    "Lior Horesh",
    "Ramón Nartallo-Kaluarachchi",
    "Shashanka Ubaru",
    "Malgorzata Zimon",
    "Ben Dongsung Huh",
    "Robert Sawko",
    "Yoshua Bengio"
  ],
  "year": 2026,
  "doi": "10.21203/rs.3.rs-9521261/v1",
  "abstract": "",
  "source": "Research Square",
  "authors_canonical": [
    "horesh_l",
    "kaluarachchi_nr",
    "shashanka_u",
    "malgorzata_z",
    "dongsung_bh",
    "robert_s",
    "yoshua_b"
  ]
}


In [19]:
# Paso 11: Eliminar duplicados
def deduplicate_publications(publications: list) -> list:
    seen_dois = set()
    unique = []

    for pub in publications:
        # deduplicar por DOI
        doi = pub.get("doi")
        if doi:
            if doi in seen_dois:
                continue
            seen_dois.add(doi)

        # deduplicar por titulo similar
        is_dup = False
        for kept in unique:
            if pub["title"] and kept["title"]:
                sim = name_similarity(pub["title"], kept["title"])
                if sim > 0.90:
                    is_dup = True
                    break

        if not is_dup:
            unique.append(pub)

    return unique

publications_clean = deduplicate_publications(publications)
print(f"antes: {len(publications)} | despues de dedup: {len(publications_clean)}")

antes: 25 | despues de dedup: 21


In [20]:
# Paso 12: Integrar pipeline completo
def author_resolution_pipeline(input_text: str) -> dict:
    # entrada: nombre libre u ORCID y salida: autor resuelto + publicaciones limpias
    print(f"procesando entrada: '{input_text}'")

    # Paso 7: resolver autor
    resolved = resolve_author(input_text, author_variants)
    print(f"autor resuelto: tipo={resolved['type']}, id={resolved.get('resolved_id', resolved.get('identifier'))}")

    # Paso 8: consultar API
    if resolved["type"] == "orcid":
        raw_pubs = fetch_publications_openalex(orcid=resolved["identifier"])
    else:
        query_name = resolved.get("best_match", {}).get("name", input_text) if resolved.get("best_match") else input_text
        raw_pubs = fetch_publications_openalex(name=query_name)

    print(f"publicaciones obtenidas: {len(raw_pubs)}")

    # Pasos 9-10: parsear y normalizar
    pubs = []
    for raw in raw_pubs:
        pub = parse_publication(raw)
        pub["abstract"] = rebuild_abstract(pub["abstract"])
        pubs.append(pub)
    pubs = normalize_publication_authors(pubs)

    # Paso 11: deduplicar
    pubs = deduplicate_publications(pubs)
    print(f"publicaciones unicas finales: {len(pubs)}")

    result = {
        "resolved_author": resolved,
        "publications": pubs
    }

    # JSON
    filename = "author_pipeline_output.json"
    with open(filename, "w", encoding="utf-8") as f:
        json.dump(result, f, ensure_ascii=False, indent=2)
    print(f"guardado en {filename}")

    return result

# Con ORCID real de Bengio
result = author_resolution_pipeline("0000-0002-9322-3515")

procesando entrada: '0000-0002-9322-3515'
autor resuelto: tipo=orcid, id=0000-0002-9322-3515
   OpenAlex ID: A5086198262 | Nombre: Yoshua Bengio
   Works encontrados: 25
publicaciones obtenidas: 25
publicaciones unicas finales: 21
guardado en author_pipeline_output.json


In [25]:
print(f"autor: {result['resolved_author']}")
print(f"\ntotal publicaciones: {len(result['publications'])}")
print("\nprimeras 3 publicaciones:")
for pub in result["publications"][:3]:
    print(f"\n{pub['title']}")
    print(f"     año: {pub['year']} | DOI: {pub['doi']}")
    print(f"     autores: {', '.join(pub['authors'][:3])}")

autor: {'type': 'orcid', 'identifier': '0000-0002-9322-3515', 'resolved_name': None}

total publicaciones: 21

primeras 3 publicaciones:

Interpretable epistemic uncertainty decomposition in sequential generative models via polynomial chaos surrogates
     año: 2026 | DOI: 10.21203/rs.3.rs-9521261/v1
     autores: Lior Horesh, Ramón Nartallo-Kaluarachchi, Shashanka Ubaru

A Recurrent Latent Variable Model for Sequential Data
     año: 2026 | DOI: 10.5281/zenodo.19983308
     autores: Jun‐Young Chung, Kyle Kastner, Laurent Dinh

Sliding Window Recurrences for Sequence Models
     año: 2025 | DOI: 10.48550/arxiv.2512.13921
     autores: Dragos Secrieru, Garyk Brixi, Yoshua Bengio
